# Notebook 04 - Gradio Demo (Singlish Marker Keyboard)

**ITI113 Team 16 - Focus C (MLOps & Deployment).** Live demo for the presentation.

Self-contained: calls the deployed serverless endpoint `iti113-team16-singlish-keyboard` and
wraps it in a Gradio UI. No dependency on `src/` or the corpus, so it runs anywhere with AWS
credentials. The endpoint currently serves the champion **Linear SVM**; this notebook is
model-agnostic and needs no change if the model is swapped again.

> Re-record the backup whenever the model behind the endpoint changes.


## 0. Install


In [1]:
!pip install -q gradio

## 1. Config


In [2]:
import json
import boto3

REGION        = "ap-southeast-1"
ENDPOINT_NAME = "iti113-team16-singlish-keyboard"

rt = boto3.client("sagemaker-runtime", region_name=REGION)
print("Calling endpoint:", ENDPOINT_NAME, "in", REGION)


Calling endpoint: iti113-team16-singlish-keyboard in ap-southeast-1


## 2. Endpoint call

Request `{"text": "..."}`; response is a one-item list like
`[{"prediction": "none", "top3": [{"marker": "none", "score": 0.53}, ...]}]`.


In [7]:
import time

def call_endpoint(text, retries=3, delay=2):
    last = None
    for attempt in range(retries):
        try:
            resp = rt.invoke_endpoint(
                EndpointName=ENDPOINT_NAME,
                ContentType="application/json",
                Body=json.dumps({"text": text}),
            )
            return json.loads(resp["Body"].read())[0]
        except Exception as e:
            last = e
            time.sleep(delay)   # wait for cold start / blip, then retry
    raise last


## 3. Label -> emoji mapping (self-contained)

The four mood families match `EMOTICON_FAMILIES` in the shared `singlish_labelling` module
(`joy, playful, sad, love`) - hard-coded here so the demo has no import dependency. Particles
render as their own word, `none` shows muted, and any unmapped label falls back to a neutral
glyph so the live demo never shows a blank.


In [4]:
# Mood families (same names as the training module) -> display glyph
FAMILY_EMOJI = {
    "joy":     "😄",
    "playful": "😜",
    "sad":     "😢",
    "love":    "❤️",
}
EMOJI_BY_GROUP = {f"emo_{fam}": glyph for fam, glyph in FAMILY_EMOJI.items()}
print("emoji groups:", EMOJI_BY_GROUP)

def pretty(label):
    """Return (glyph, caption). glyph empty for text-only markers (particles / none)."""
    if label is None or label == "none":
        return "", "no marker"
    if str(label).startswith("emo_"):
        return EMOJI_BY_GROUP.get(label, "🙂"), label.replace("emo_", "")
    return "", str(label)  # particle: the word itself is the suggestion

def render_label(label):
    glyph, caption = pretty(label)
    if glyph:
        return f"{glyph}  {caption}"
    return caption if label != "none" else "— none"


emoji groups: {'emo_joy': '😄', 'emo_playful': '😜', 'emo_sad': '😢', 'emo_love': '❤️'}


## 4. Predict function for the UI


In [8]:
def predict_marker(text):
    text = (text or "").strip()
    if not text:
        return "Type a message first.", {}
    try:
        result = call_endpoint(text)
    except Exception as e:
        return f"Endpoint error: {e}", {}
    top_label = result.get("prediction", "none")
    glyph, caption = pretty(top_label)
    headline = f"{glyph}  {caption}".strip() if glyph else f"Suggested marker: {caption}"
    top3 = result.get("top3", [])
    scores = {render_label(d["marker"]): float(d["score"]) for d in top3}
    if not scores:
        scores = {render_label(top_label): 1.0}
    return headline, scores


## 5. Gradio UI


In [9]:
import gradio as gr

with gr.Blocks(title="Team 16 - Singlish Marker Keyboard", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## Singlish Marker Keyboard\n"
        "Type a message. The model predicts the sentence-final marker: a Singlish particle "
        "(`lah` / `leh` / `lor`), a mood-grouped emoji, or none. The 3-suggestion bar is the "
        "product view - the marker is usually one tap away even when `none` leads."
    )
    inp = gr.Textbox(label="Your message", placeholder="e.g. i love you", lines=2)
    btn = gr.Button("Predict marker", variant="primary")
    out_head = gr.Textbox(label="Top suggestion", interactive=False)
    out_top3 = gr.Label(label="Top-3 suggestions", num_top_classes=3)
    btn.click(predict_marker, inputs=inp, outputs=[out_head, out_top3])
    inp.submit(predict_marker, inputs=inp, outputs=[out_head, out_top3])
    gr.Examples(
        ["i love you", "goodnight", "so sad fever", "dunno leh", "ok can", "so happy"],
        inputs=inp,
    )

demo.launch(share=True)


/tmp/ipykernel_4905/1307724071.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Team 16 - Singlish Marker Keyboard", theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://57bc67b4c80c465319.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 6. After it works

1. Record the backup screen capture (mandatory fallback).
2. Note the `share=True` public URL for the report / ZIP.
3. Link expires ~1 week and dies when the kernel stops; regenerate near the presentation.
